In [2]:
import pandas as pd
from scipy.stats import chi2_contingency

# 0. Cargar datos (el notebook está en notebooks/, los CSV en data/)
mql = pd.read_csv("../data/olist_marketing_qualified_leads_dataset.csv",
                  parse_dates=["first_contact_date"])
deals = pd.read_csv("../data/olist_closed_deals_dataset.csv",
                    parse_dates=["won_date"])

df = mql.merge(deals, on="mql_id", how="left")
df["convirtio"] = df["won_date"].notna().astype(int)

# 1. Unir nulos con unknown
df["origin"] = df["origin"].fillna("unknown")

# 2. Tabla de conversión por canal
conv_canal = (df.groupby("origin")["convirtio"]
                .agg(leads="count", cierres="sum", tasa="mean")
                .sort_values("tasa", ascending=False))
print(conv_canal)

# 3. ¿Son significativas las diferencias entre canales?
tabla = pd.crosstab(df["origin"], df["convirtio"])
chi2, p, _, _ = chi2_contingency(tabla)
print(f"\nchi2 = {chi2:.2f} | p-value = {p:.6f}")

# 4. Landing pages: cuántas hay y cuánto convierten
lp = (df.groupby("landing_page_id")["convirtio"]
        .agg(leads="count", cierres="sum", tasa="mean")
        .sort_values("leads", ascending=False))
print(f"\nLanding pages distintas: {lp.shape[0]}")
print(lp.head(10))
print("\nLeads por landing page (distribución):")
print(lp["leads"].describe())

# 5. Tiempo hasta el cierre (solo convertidos)
cerrados = df[df["convirtio"] == 1].copy()
cerrados["dias_cierre"] = (cerrados["won_date"] - cerrados["first_contact_date"]).dt.days
print("\nDías hasta el cierre:")
print(cerrados["dias_cierre"].describe())

# 6. Leads y conversión por mes
df["mes"] = df["first_contact_date"].dt.to_period("M")
print("\nLeads y conversión por mes:")
print(df.groupby("mes")["convirtio"].agg(leads="count", cierres="sum", tasa="mean"))

                   leads  cierres      tasa
origin                                     
unknown             1159      193  0.166523
paid_search         1586      195  0.122951
organic_search      2296      271  0.118031
direct_traffic       499       56  0.112224
referral             284       24  0.084507
social              1350       75  0.055556
display              118        6  0.050847
other_publicities     65        3  0.046154
email                493       15  0.030426
other                150        4  0.026667

chi2 = 137.69 | p-value = 0.000000

Landing pages distintas: 495
                                  leads  cierres      tasa
landing_page_id                                           
b76ef37428e6799c421989521c0e5077    912      171  0.187500
22c29808c4f815213303f8933030604c    883      174  0.197055
58326e62183c14b0c03085c33b9fdc44    495       27  0.054545
88740e65d5d6b056e0cda098e1ea6313    445       31  0.069663
ce1a65abd0973638f1c887a6efcfa82d    394       59  0.

In [3]:
# 1. Chequeos de calidad
cerrados = df[df["convirtio"] == 1].copy()
cerrados["dias_cierre"] = (cerrados["won_date"] - cerrados["first_contact_date"]).dt.days
print((cerrados["dias_cierre"] < 0).sum(), "cierres con fecha anterior al primer contacto")
print("Contacto:", df["first_contact_date"].min(), "->", df["first_contact_date"].max())
print("Cierre:  ", df["won_date"].min(), "->", df["won_date"].max())
print("landing_page_id nulos:", df["landing_page_id"].isna().sum())

# 2. Agrupar landing pages: top 10 + "LP_otras"
lp_counts = df["landing_page_id"].value_counts()
top_lp = lp_counts.head(10).index
mapa = {lp_id: f"LP_{i+1}" for i, lp_id in enumerate(top_lp)}
df["lp_grupo"] = df["landing_page_id"].map(mapa).fillna("LP_otras")

print(df.groupby("lp_grupo")["convirtio"]
        .agg(leads="count", cierres="sum", tasa="mean")
        .sort_values("leads", ascending=False))

# 3. Cruce canal x landing page (volumen de leads)
print(pd.crosstab(df["origin"], df["lp_grupo"]))

# 4. Construir el journey de cada lead
df["estado_final"] = df["convirtio"].map({1: "Conversion", 0: "Sin_conversion"})
df["journey"] = df.apply(
    lambda r: ["Inicio", r["origin"], r["lp_grupo"], r["estado_final"]], axis=1
)
print(df["journey"].head())

1 cierres con fecha anterior al primer contacto
Contacto: 2017-06-14 00:00:00 -> 2018-05-31 00:00:00
Cierre:   2017-12-05 02:00:00 -> 2018-11-14 18:04:19
landing_page_id nulos: 0
          leads  cierres      tasa
lp_grupo                          
LP_otras   3669      260  0.070864
LP_1        912      171  0.187500
LP_2        883      174  0.197055
LP_3        495       27  0.054545
LP_4        445       31  0.069663
LP_5        394       59  0.149746
LP_6        330       67  0.203030
LP_7        310       21  0.067742
LP_8        291       10  0.034364
LP_9        156       18  0.115385
LP_10       115        4  0.034783
lp_grupo           LP_1  LP_10  LP_2  LP_3  LP_4  LP_5  LP_6  LP_7  LP_8  \
origin                                                                     
direct_traffic       25      4   105     5     3    46     5    20    22   
display               3      0     4     0     1     6     0     3     4   
email                 3      3    11     0     1     8     5  

In [4]:
import numpy as np
from collections import Counter

# 1. Conversión a ventana fija de 90 días (comparable entre meses)
df["dias"] = (df["won_date"] - df["first_contact_date"]).dt.days.clip(lower=0)
df["conv_90"] = ((df["convirtio"] == 1) & (df["dias"] <= 90)).astype(int)
df["mes"] = df["first_contact_date"].dt.to_period("M")
print(df.groupby("mes")[["convirtio", "conv_90"]].mean())
print("Cierres después de 90 días:", ((df["convirtio"] == 1) & (df["dias"] > 90)).sum())

# 2. ¿El canal importa dentro de una misma landing page?
top = df[df["lp_grupo"].isin(["LP_1", "LP_2", "LP_5", "LP_6"])]
print(top.groupby(["lp_grupo", "origin"])["convirtio"]
         .agg(leads="count", tasa="mean")
         .query("leads >= 30"))

# 3. Cadena de Markov con efecto de remoción
def conteos(journeys):
    c = Counter()
    for j in journeys:
        for a, b in zip(j[:-1], j[1:]):
            c[(a, b)] += 1
    return c

def p_conversion(counts, removed=None):
    estados = sorted({s for par in counts for s in par} - {"Conversion", "Sin_conversion"})
    idx = {s: i for i, s in enumerate(estados)}
    n = len(estados)
    Q, R, tot = np.zeros((n, n)), np.zeros(n), np.zeros(n)
    for (a, b), v in counts.items():
        tot[idx[a]] += v
        if b == removed or b == "Sin_conversion":
            continue                      # el flujo se pierde (estado nulo)
        if b == "Conversion":
            R[idx[a]] += v
        else:
            Q[idx[a], idx[b]] += v
    Q, R = Q / tot[:, None], R / tot
    return np.linalg.solve(np.eye(n) - Q, R)[idx["Inicio"]]

cnt = conteos(df["journey"])
base = p_conversion(cnt)
print("P(conversión) base:", round(base, 4), "| observada:", round(df["convirtio"].mean(), 4))

canales = df["origin"].unique()
efecto = {c: (base - p_conversion(cnt, removed=c)) / base for c in canales}
markov = pd.Series(efecto)
markov = markov / markov.sum()            # normalizado a 100%

# 4. Comparación con first-touch
first_touch = df.groupby("origin")["convirtio"].sum()
first_touch = first_touch / first_touch.sum()

comp = pd.DataFrame({"Markov": markov, "First-touch": first_touch}).round(4)
comp["diferencia"] = (comp["Markov"] - comp["First-touch"]).round(4)
print(comp.sort_values("Markov", ascending=False))

         convirtio   conv_90
mes                         
2017-06   0.000000  0.000000
2017-07   0.008368  0.000000
2017-08   0.023316  0.000000
2017-09   0.022436  0.000000
2017-10   0.033654  0.000000
2017-11   0.040449  0.017978
2017-12   0.055000  0.030000
2018-01   0.133216  0.113059
2018-02   0.144942  0.124514
2018-03   0.142249  0.120102
2018-04   0.135355  0.126479
2018-05   0.099770  0.083653
Cierres después de 90 días: 150
                         leads      tasa
lp_grupo origin                         
LP_1     organic_search    116  0.206897
         paid_search        81  0.086420
         unknown           657  0.203957
LP_2     direct_traffic    105  0.171429
         organic_search    495  0.226263
         paid_search       124  0.161290
         referral           56  0.178571
         social             33  0.030303
         unknown            41  0.195122
LP_5     direct_traffic     46  0.239130
         organic_search    232  0.146552
         paid_search        5

In [5]:
# Periodo de análisis: desde enero 2018
df_a = df[df["first_contact_date"] >= "2018-01-01"].copy()
print(df_a.shape, df_a["convirtio"].mean().round(4))

# Guardar dataset procesado (data/ del repositorio)
cols = ["mql_id", "first_contact_date", "origin", "landing_page_id", "lp_grupo",
        "convirtio", "conv_90", "won_date"]
df_a[cols].to_csv("../data/leads_procesados.csv", index=False)

(5998, 24) 0.1302


In [7]:
df = pd.read_csv("../data/leads_procesados.csv",
                 parse_dates=["first_contact_date", "won_date"])

In [8]:
comp.to_csv("../data/atribucion_real.csv")

In [10]:
journeys_a = [["Inicio", o, lp, "Conversion" if c == 1 else "Sin_conversion"]
              for o, lp, c in zip(df_a["origin"], df_a["lp_grupo"], df_a["conv_90"])]

efecto = efecto_remocion(journeys_a)          # incluye canales y landing pages

canales = df_a["origin"].unique()
paginas = df_a["lp_grupo"].unique()

markov_canal = efecto[canales] / efecto[canales].sum()
markov_lp = efecto[paginas] / efecto[paginas].sum()

ft = df_a.groupby("origin")["conv_90"].sum()
ft = ft / ft.sum()

comp_a = pd.DataFrame({"Markov": markov_canal, "First-touch": ft}).round(4)
print(comp_a.sort_values("Markov", ascending=False))
print(markov_lp.round(4).sort_values(ascending=False))

comp_a.to_csv("../data/atribucion_real.csv")
markov_lp.rename("Markov").to_csv("../data/atribucion_lp.csv")

                   Markov  First-touch
organic_search     0.3158       0.3142
paid_search        0.2096       0.2168
unknown            0.2035       0.2493
social             0.1011       0.0811
direct_traffic     0.0659       0.0723
email              0.0422       0.0177
referral           0.0352       0.0310
display            0.0102       0.0074
other              0.0097       0.0059
other_publicities  0.0068       0.0044
LP_otras    0.2802
LP_1        0.2242
LP_2        0.2198
LP_6        0.0855
LP_5        0.0693
LP_3        0.0324
LP_4        0.0310
LP_7        0.0236
LP_9        0.0162
LP_8        0.0118
LP_10       0.0059
dtype: float64
